In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os

# -----------------------------------
# MySQL connection details
# -----------------------------------

username = "root"
password = "**************"
host = "localhost"
port = "3306"
database = "ecommerce_analytics"

# Create MySQL connection
engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}"
)


# -----------------------------------
# Dataset folder
# -----------------------------------

dataset_path = r"D:\SQL_FOLDER\Dataset"


# -----------------------------------
# CSV file → MySQL table
# -----------------------------------

files = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_sellers_dataset.csv": "sellers",
    "olist_products_dataset.csv": "products",
    "product_category_name_translation.csv": "product_category_name_translation",
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews"
}


# -----------------------------------
# Tables already loaded
# -----------------------------------

loaded_tables = {
    "customers"
}


# -----------------------------------
# Load CSV files
# -----------------------------------

for file_name, table_name in files.items():

    # Skip tables that are already loaded
    if table_name in loaded_tables:
        print(f"\nSkipping '{table_name}' - already loaded.")
        continue

    file_path = os.path.join(dataset_path, file_name)

    print("\n" + "=" * 50)
    print(f"Loading: {file_name}")
    print(f"Table: {table_name}")
    print("=" * 50)

    # Read CSV
    df = pd.read_csv(file_path)

    print(f"Rows found: {len(df):,}")

    # -----------------------------------
    # Fix product column names
    # -----------------------------------

    if table_name == "products":

        df = df.rename(columns={
            "product_name_lenght": "product_name_length",
            "product_description_lenght": "product_description_length"
        })

    # -----------------------------------
    # Insert data in chunks
    # -----------------------------------

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="append",
        index=False,
        chunksize=5000
    )

    print(f"✓ {len(df):,} rows loaded into '{table_name}'")


print("\n" + "=" * 50)
print("All required files have been processed!")
print("=" * 50)

In [5]:
file_path = r"D:\SQL_FOLDER\Dataset\olist_order_reviews_dataset.csv"

df_reviews = pd.read_csv(file_path)

print(f"Total rows: {len(df_reviews):,}")

duplicate_count = df_reviews["review_id"].duplicated().sum()

print(f"Duplicate review_id values: {duplicate_count}")

Total rows: 99,224
Duplicate review_id values: 814


In [6]:
# Number of duplicate rows when considering all columns
exact_duplicates = df_reviews.duplicated().sum()

print(f"Exact duplicate rows: {exact_duplicates}")

Exact duplicate rows: 0


In [8]:
duplicate_pairs = df_reviews.duplicated(
    subset=["review_id", "order_id"]
).sum()

print(f"Duplicate review_id + order_id pairs: {duplicate_pairs}")

Duplicate review_id + order_id pairs: 0


In [9]:
df_reviews.to_sql(
    name="order_reviews",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000
)

print(f"✓ {len(df_reviews):,} review rows loaded successfully!")

✓ 99,224 review rows loaded successfully!
